# Avance Fase 3 — Semana 2 (Formativa)

**Grupo 4 · MCDI500**

Este notebook parte del conjunto ya limpio y validado en la Fase 2
(`data/processed/ens_procesado.csv`). No se rehace la limpieza ni se
agregan datos nuevos: el objetivo de esta fase es reorganizar el
código que ya funciona, medir su eficiencia, y decidir si el
proyecto justifica recursividad.

In [ ]:
import sys
import time
import pandas as pd
from pathlib import Path

SEMILLA = 2026

def encontrar_raiz_proyecto(marcador=".git"):
    """Sube por las carpetas padre hasta encontrar la raiz del repositorio."""
    actual = Path.cwd()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / marcador).exists():
            return carpeta
    raise FileNotFoundError(f"No se encontro '{marcador}' en ningun directorio padre")

RAIZ = encontrar_raiz_proyecto()
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("Raiz del proyecto:", RAIZ)

In [ ]:
ARCHIVO = RAIZ / "data" / "processed" / "ens_procesado.csv"
df = pd.read_csv(ARCHIVO)

print(f"Archivo: {ARCHIVO}")
print(f"Conjunto cargado: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()

## El conjunto antes de tocarlo

Antes de escribir cualquier función, revisamos brevemente el estado
del conjunto que ya dejó lista la Fase 2. No se modifica nada aquí:
es solo el punto de partida para las mediciones de esta fase.

In [ ]:
print("Dimensiones:", df.shape)
print("\nTipos de datos:")
print(df.dtypes.value_counts())
print("\nValores nulos totales:", int(df.isnull().sum().sum()))
print("\nColumnas con nulos declarados (GPAQ imputada, as27/as28 con NaN genuino):")
print(df.isnull().sum()[df.isnull().sum() > 0])

## 3. Medición de eficiencia: búsqueda por posición

`IdEncuesta` se excluyó del conjunto procesado en la Fase 2, por no
aportar valor analítico. Para esta medición trabajamos con el
índice propio del DataFrame como identificador de cada persona,
comparando dos formas de ubicar un registro: recorriendo el
conjunto fila por fila frente a acceder directamente por posición.

In [ ]:
import sys
sys.path.append(str(RAIZ / "F3" / "src"))

from busqueda import buscar_recorriendo, buscar_por_indice, medir_tiempo

print("Funciones importadas correctamente desde F3/src/busqueda.py")

## 3.1 Medición: tiempo de cada versión

Aplicamos las tres reglas para que la medición valga: repetir y
conservar el mínimo, medir sobre el tamaño real del conjunto
(5.511 filas), y comprobar que ambas versiones entregan el mismo
resultado antes de comparar los tiempos.


In [ ]:
# posicion_prueba se define aqui mismo, sin depender de celdas anteriores
posicion_prueba = df.shape[0] - 1

tiempo_a, resultado_a = medir_tiempo(buscar_recorriendo, df, posicion_prueba)
tiempo_b, resultado_b = medir_tiempo(buscar_por_indice, df, posicion_prueba)

print(f"Recorriendo:  {tiempo_a:.6f} s")
print(f"Por indice:   {tiempo_b:.6f} s")
print(f"El acceso por indice es {tiempo_a / tiempo_b:.1f} veces mas rapido")

assert resultado_a.equals(resultado_b), "Las dos versiones no coinciden"
print("\nVerificado: ambas versiones entregan exactamente el mismo resultado.")

## 3.2 Interpretación

El acceso por posición (`buscar_por_indice`) resultó **1.764 veces
más rápido** que recorrer el conjunto fila por fila
(`buscar_recorriendo`), midiendo sobre el peor caso posible (la
última fila del conjunto).

La diferencia se explica por la naturaleza de cada operación:
`buscar_recorriendo` tiene una complejidad temporal de **O(n)**, ya
que en el peor caso revisa las 5.511 filas una por una antes de
encontrar la buscada. `buscar_por_indice`, en cambio, es
prácticamente **O(1)**: pandas accede directamente a la posición de
memoria correspondiente, sin recorrer nada.

Esta diferencia se replica en el propio pipeline del proyecto: por
ejemplo, en la Fase 2 se verificó la unicidad de `IdEncuesta` y se
cruzaron los valores de `as27` y `as28` entre sí. Un enfoque que
recorra el conjunto completo cada vez que se necesita ubicar un
registro se vuelve costoso a medida que crece la muestra; indexar
primero (como ya se hace de forma nativa en pandas con `.loc`/`.iloc`)
es la opción adoptada para cualquier búsqueda repetida sobre el
conjunto.

## 4. ¿Necesita este proyecto recursividad?

Antes de forzar una recursión artificial, evaluamos si el pipeline
de la Fase 2 tiene un problema que la justifique.

In [ ]:
# Revision de los pasos del pipeline de Fase 2:
pasos_pipeline = [
    "seleccionar_variables_ens",
    "filtrar_ponderador_valido",
    "explorar_dataframe",
    "revisar_codigos_especiales",
    "marcar_codigos_no_respuesta",
    "imputar_nulos_numericos / imputar_nulos_categoricos",
    "codificar_one_hot",
    "escalar_caracteristicas",
    "validar_dataset",
]

print(f"El pipeline tiene {len(pasos_pipeline)} pasos, en una secuencia FIJA y conocida:")
for i, paso in enumerate(pasos_pipeline, 1):
    print(f"  {i}. {paso}")

## 4.1 Conclusión: no se justifica recursión en el pipeline principal

El pipeline principal tiene una secuencia fija y conocida de nueve
pasos, sin niveles de profundidad variable ni una estructura
autosimilar. Un enfoque iterativo (aplicar cada función en orden)
es preferible: es más simple, más legible, y no arriesga agotar la
pila de llamadas de Python.

La única función recursiva presente en el proyecto es `aplanar()`,
utilizada en la Fase 1 para convertir el diccionario anidado de
metadatos del proyecto en pares clave-valor. Esa función sí se
justifica, porque la profundidad de anidamiento del diccionario no
se conoce de antemano al escribir el código: podría tener uno,
dos o más niveles, y la recursión se adapta a cualquiera de ellos
sin necesidad de escribir un bucle distinto para cada caso.